# Advanced Model: Deep Learning (Keras / TensorFlow)

Da die PyTorch-Installation auf Kaggle für deinen GPU-Typ aktuell einen Bug hat, weichen wir auf **TensorFlow/Keras** aus.
TensorFlow ist direkt von Google (den Betreibern von Kaggle) und funktioniert zu 100 % nahtlos und extrem stabil auf allen Kaggle GPUs (T4x2, P100) und sogar TPUs!

Wir bauen exakt dieselbe Architektur: **1D-CNN + Bidirektionales LSTM**.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization, Activation, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow Version:", tf.__version__)
print("Gefundene GPUs:", len(tf.config.list_physical_devices('GPU')))

## 1. Daten Laden & Skalieren

In [ ]:
KAGGLE_PATH = '/kaggle/input/datasets/axxtur/nycu-data-mining-assignment-3'
if not os.path.exists(KAGGLE_PATH):
    KAGGLE_PATH = '/kaggle/input/nycu-data-mining-assignment-3'
    if not os.path.exists(KAGGLE_PATH):
        KAGGLE_PATH = 'nycu-data-mining-assignment-3'

print("Lade Daten...")
train_data = np.load(os.path.join(KAGGLE_PATH, 'train_data.npz'), allow_pickle=True)
X_train_raw = train_data['X'] # (11020, 300, 6)
y_train = train_data['y']
file_ids_train = train_data['file_ids']
user_ids_train = train_data['user_ids']

test_data = np.load(os.path.join(KAGGLE_PATH, 'test_data.npz'), allow_pickle=True)
X_test_raw = test_data['X']
file_ids_test = test_data['file_ids']

# TensorFlow mag die Shape (Batch, Timesteps, Channels) -> (11020, 300, 6)
# Wir müssen nichts drehen (kein permute)!

# Skalierung (Standardisierung ist Pflicht für Neural Networks!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw.reshape(-1, 6)).reshape(X_train_raw.shape)
X_test_scaled = scaler.transform(X_test_raw.reshape(-1, 6)).reshape(X_test_raw.shape)

## 2. Modell Architektur definieren (Keras)

In [ ]:
def create_model():
    model = Sequential([
        # 1D Convolution Block 1
        Conv1D(64, kernel_size=9, padding='same', input_shape=(300, 6)),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        
        # 1D Convolution Block 2
        Conv1D(128, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        
        # 1D Convolution Block 3
        Conv1D(256, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        
        # Bi-direktionales LSTM
        Bidirectional(LSTM(128)),
        
        # Fully Connected Classifier
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(6, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
        loss='sparse_categorical_crossentropy', 
        metrics=['sparse_categorical_accuracy']
    )
    return model

create_model().summary()

## 3. Training mit Cross-Validation

In [ ]:
gkf = GroupKFold(n_splits=5)
test_preds_proba = np.zeros((len(X_test_scaled), 6))
cv_scores = []

# Klassengewichtung für seltene Klassen (Label 4!)
from sklearn.utils.class_weight import compute_class_weight
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights_array)}

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_scaled, y_train, groups=user_ids_train)):
    print(f"\n{'='*30}\nStart Fold {fold+1}\n{'='*30}")
    
    X_tr, y_tr = X_train_scaled[train_idx], y_train[train_idx]
    X_va, y_va = X_train_scaled[val_idx], y_train[val_idx]
    
    model = create_model()
    
    # Callbacks: Stoppe wenn sich Val-Loss nicht mehr verbessert, verringere Learning Rate
    early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    
    # Training (Keras übernimmt alles auf der GPU automatisch)
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=25,
        batch_size=128,
        class_weight=class_weights_dict,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    # F1-Score ausrechnen
    val_preds = np.argmax(model.predict(X_va, verbose=0), axis=1)
    fold_f1 = f1_score(y_va, val_preds, average='macro')
    cv_scores.append(fold_f1)
    print(f"\n-> Fold {fold+1} Best F1-Macro: {fold_f1:.4f}")
    
    # Mache direkt Predictions auf dem Test-Set fürs Ensemble
    test_preds_proba += model.predict(X_test_scaled, verbose=0) / 5.0

print(f"\nOverall Cross-Validation F1-Macro: {np.mean(cv_scores):.4f}")

## 4. Submission erstellen

In [ ]:
final_preds = np.argmax(test_preds_proba, axis=1)

submission = pd.DataFrame({
    'Id': file_ids_test,
    'Label': final_preds
})

submission.to_csv('submission_tf.csv', index=False)
print("Saved submission_tf.csv! Ready for Kaggle upload.")
submission.head()